In [1]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.cluster import KMeans

In [ ]:
print("--- INICIANDO FASE 2: MODELO DE CLASIFICACIÓN ---")

# 1. Cargamos el ds ya pulido
df = pd.read_csv('dataset_preparado.csv')

# 2. Separamos las características (X) de la etiqueta objetivo (y)
# IMPORTANTE: Eliminamos 'price' para evitar "Data Leakage"
X = df.drop(['categoria_precio', 'price'], axis=1)
y = df['categoria_precio']

# 3. Cortamos el dataset (80% para entrenar, 20% para el examen oculto)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. Escalamos los datos para nivelar el terreno matemático
scaler = StandardScaler()
# Fiteamos y transformamos los datos de entrenamiento
X_train_scaled = scaler.fit_transform(X_train)
# SOLO transformamos los de testeo (sin fitear) para no "espiar" la métrica
X_test_scaled = scaler.transform(X_test) 

# 5. Instanciamos y entrenamos el Motor
modelo_clasificacion = LogisticRegression(max_iter=1000)
modelo_clasificacion.fit(X_train_scaled, y_train)

# 6. Examen con el 20% que nunca vio
predicciones = modelo_clasificacion.predict(X_test_scaled)
precision = accuracy_score(y_test, predicciones)

print(f"✅ Modelo entrenado y testeado con éxito.")
print(f"🎯 Precisión del modelo (Accuracy): {precision * 100:.2f}%\n")

print("--- REPORTE DETALLADO (Métricas Profesionales) ---")
print(classification_report(y_test, predicciones))

--- INICIANDO FASE 2: MODELO DE CLASIFICACIÓN ---
✅ Modelo entrenado y testeado con éxito.
🎯 Precisión del modelo (Accuracy): 92.68%

--- REPORTE DETALLADO (Métricas Profesionales) ---
              precision    recall  f1-score   support

           0       0.95      0.91      0.93        23
           1       0.89      0.94      0.92        18

    accuracy                           0.93        41
   macro avg       0.92      0.93      0.93        41
weighted avg       0.93      0.93      0.93        41



In [ ]:
print("--- INICIANDO FASE 2: MODELO DE REGRESIÓN ---")

# 1. Definimos nuestra nueva respuesta correcta (El precio exacto)
y_regresion = df['price']

# 2. Volvemos a cortar el dataset
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X, y_regresion, test_size=0.2, random_state=42
)

# 3. Escalamos los datos
X_train_reg_scaled = scaler.fit_transform(X_train_reg)
X_test_reg_scaled = scaler.transform(X_test_reg)

# 4. Instanciamos y entrenamos el Bosque Aleatorio
modelo_regresion = RandomForestRegressor(n_estimators=100, random_state=42)
modelo_regresion.fit(X_train_reg_scaled, y_train_reg)

# 5. Le tomamos el examen al modelo
predicciones_precio = modelo_regresion.predict(X_test_reg_scaled)

# 6. Evaluamos
mae = mean_absolute_error(y_test_reg, predicciones_precio)
r2 = r2_score(y_test_reg, predicciones_precio)

print(f"✅ Modelo de Regresión entrenado.")
print(f"📉 Margen de error promedio (MAE): ${mae:.2f}")
print(f"🎯 Nivel de confianza (R2 Score): {r2:.4f}\n")

print("--- PRUEBA DE FUEGO (Real vs Predicción) ---")
# Comparamos 5 autos al azar para ver qué tan cerca estuvo
prueba = pd.DataFrame({
    'Precio Real': y_test_reg.values[:5],
    'Predicción del Modelo': predicciones_precio[:5]
})
prueba['Diferencia'] = (prueba['Precio Real'] - prueba['Predicción del Modelo']).abs()
print(prueba)

--- INICIANDO FASE 2: MODELO DE REGRESIÓN ---
✅ Modelo de Regresión entrenado.
📉 Margen de error promedio (MAE): $1286.40
🎯 Nivel de confianza (R2 Score): 0.9577

--- PRUEBA DE FUEGO (Real vs Predicción) ---
   Precio Real  Predicción del Modelo   Diferencia
0    30760.000           36029.305000  5269.305000
1    17859.167           19176.280000  1317.113000
2     9549.000            9068.033333   480.966667
3    11850.000           13127.408333  1277.408333
4    28248.000           27206.100000  1041.900000


In [ ]:
print("--- EXPORTANDO MODELOS PARA PRODUCCIÓN ---")

# 1. Exportamos el modelo de Clasificación
joblib.dump(modelo_clasificacion, 'modelo_clasificacion.pkl')

# 2. Exportamos el modelo de Regresión
joblib.dump(modelo_regresion, 'modelo_regresion.pkl')

# 3. Exportamos el Scaler
joblib.dump(scaler, 'scaler.pkl')

# 4. Guardamos la lista de columnas exactas que espera el modelo
columnas_entrenamiento = X.columns.tolist()
joblib.dump(columnas_entrenamiento, 'columnas_modelo.pkl')

print("✅ Archivos generados exitosamente:")
print("- modelo_clasificacion.pkl")
print("- modelo_regresion.pkl")
print("- scaler.pkl")
print("- columnas_modelo.pkl")

--- EXPORTANDO MODELOS PARA PRODUCCIÓN ---
✅ Archivos generados exitosamente:
- modelo_clasificacion.pkl
- modelo_regresion.pkl
- scaler.pkl
- columnas_modelo.pkl


In [ ]:
print("--- INICIANDO MODELO DE CLUSTERING (K-MEANS) ---")

# 1. Usamos todo tu dataset original de características
X_completo_scaled = scaler.transform(X)

# 2. Instanciamos K-Means. Elegimos 4 grupos (clústeres) como punto de partida.
modelo_kmeans = KMeans(n_clusters=4, random_state=42, n_init='auto')

# 3. Entrenamos el modelo (Fijate que NO le pasamos ninguna "y", solo la "X")
modelo_kmeans.fit(X_completo_scaled)

# 4. Vemos cómo asignó los autos a los grupos
etiquetas_grupos = modelo_kmeans.labels_
print("✅ Modelo K-Means entrenado exitosamente.")
print(f"Los primeros 10 autos fueron asignados a estos grupos: {etiquetas_grupos[:10]}")

# 5. Exportamos el modelo para usarlo en la API
joblib.dump(modelo_kmeans, 'Modelos/modelo_clustering.pkl')

print("✅ Archivo 'modelo_clustering.pkl' guardado en la carpeta /Modelos.")

--- INICIANDO MODELO DE CLUSTERING (K-MEANS) ---
✅ Modelo K-Means entrenado exitosamente.
Los primeros 10 autos fueron asignados a estos grupos: [3 3 3 2 2 2 2 2 2 2]
✅ Archivo 'modelo_clustering.pkl' guardado en la carpeta /Modelos.


In [ ]:
print("--- PERFILANDO LA IDENTIDAD DE LOS CLÚSTERES ---")

# 1. Le pegamos las etiquetas (0, 1, 2, 3) a tu dataset limpio original
df['Cluster'] = modelo_kmeans.labels_

# 2. Elegimos qué columnas queremos "espiar" para entender el grupo
columnas_para_espiar = ['price', 'horsepower', 'curbweight', 'highwaympg']

# 3. Agrupamos por clúster y calculamos el promedio (mean)
perfil_grupos = df.groupby('Cluster')[columnas_para_espiar].mean()

# Imprimimos la tabla para que sea legible
print(perfil_grupos.round(2))

--- PERFILANDO LA IDENTIDAD DE LOS CLÚSTERES ---
            price  horsepower  curbweight  highwaympg
Cluster                                              
0         8256.05       79.12     2187.57       35.21
1        20638.81      123.00     3273.75       25.38
2        15645.62      118.64     2703.30       27.09
3        20569.90      158.36     2907.16       25.52
